# On the very first run, the pipeline performs a FULL load; all subsequent runs switch to INCREMENTAL mode.

In [0]:
# External libraries for HTTP requests and retries
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# Standard libraries for time, JSON, and datetime handling
import time
import json
import io
from datetime import datetime, timezone
from urllib.parse import urlparse, unquote

# Type hints for better code clarity and static analysis
from typing import Optional, List, Dict

# Data handling
import pandas as pd
from pyspark.sql import Row

In [0]:
# ================= SECRETS =================

# TopDesk
topdesk_username = "topdeskapi"
topdesk_password = dbutils.secrets.get(scope="topdesk-scope", key="topdeskapi")

# Jira
jira_username = dbutils.secrets.get(scope="jira-scope", key="jira-username")
jira_token    = dbutils.secrets.get(scope="jira-scope", key="jira-token")

# Microsoft Entra (Graph API)
client_id     = dbutils.secrets.get(scope="microsoft_graph-scope", key="client-id")
client_secret = dbutils.secrets.get(scope="microsoft_graph-scope", key="client-secret")
tenant_id     = dbutils.secrets.get(scope="microsoft_graph-scope", key="tenant-id")

# Tempo
tempo_token = dbutils.secrets.get(scope="tempo-scope", key="tempo-token")

# ================= CONFIG =================
# 
CATALOG        = "elixir"
BRONZE_SCHEMA  = "brz"
META_SCHEMA    = "meta"
BATCH_SIZE     = 10_000
PAGE_SIZE      = 100
MAX_RETRIES    = 3
REQUEST_TIMEOUT = 30
LOAD_TYPE      = "INCREMENTAL"   # "FULL" or "INCREMENTAL"
STATE_TABLE    = f"{CATALOG}.{META_SCHEMA}.pipeline_state"
SHAREPOINT_HOST = "msfintl.sharepoint.com"

# ================= ENDPOINTS =================
# Each endpoint declares its own source (for auth) and pagination strategy.
# Supported sources:     topdesk | jira | entra | tempo | sharepoint
# Supported pagination:  SINGLE | OFFSET | NEXT | JIRA_NEXT | GRAPH_NEXT | ODATA_NEXT | TEMPO_NEXT | SHAREPOINT_FILE

TOPDESK_BASE       = "https://elixir.msf.org/tas/api"
TOPDESK_ODATA_BASE = "https://msf.topdesk.net/services/reporting/v2/odata"

# Shared SharePoint config for all Excel files
_SP_SITE = "sites/GRP-SITS-COSTRECOVERY"
_SP_DIR  = "General/PowerBI Master Files"

ENDPOINTS = [
    # ── TopDesk REST API ────────────────────────────────────
    {
        "name": "incidents",
        "source": "topdesk",
        "url": f"{TOPDESK_BASE}/incidents",
        "pagination": "OFFSET",
        "incremental_field": "modificationDate",
        "bronze_table": f"{CATALOG}.{BRONZE_SCHEMA}.topdesk_incidents_raw",
    },
    # {
    #     "name": "time_registrations",
    #     "source": "topdesk",
    #     "url": f"{TOPDESK_BASE}/incidents/timeregistrations",
    #     "pagination": "NEXT",
    #     "incremental_field": "modificationDate",
    #     "bronze_table": f"{CATALOG}.{BRONZE_SCHEMA}.topdesk_timeregistrations_raw",
    #     "params": {"fields": "timeSpent,operator.id,creationDate,parent.id,notes,modificationDate"},
    # },
    # {
    #     "name": "priorities",
    #     "source": "topdesk",
    #     "url": f"{TOPDESK_BASE}/incidents/priorities",
    #     "pagination": "SINGLE",
    #     "bronze_table": f"{CATALOG}.{BRONZE_SCHEMA}.topdesk_incident_priorities_raw",
    # },
    # {
    #     "name": "statuses",
    #     "source": "topdesk",
    #     "url": f"{TOPDESK_BASE}/incidents/statuses",
    #     "pagination": "SINGLE",
    #     "bronze_table": f"{CATALOG}.{BRONZE_SCHEMA}.topdesk_incident_statuses_raw",
    # },
    # {
    #     "name": "urgencies",
    #     "source": "topdesk",
    #     "url": f"{TOPDESK_BASE}/incidents/urgencies",
    #     "pagination": "SINGLE",
    #     "bronze_table": f"{CATALOG}.{BRONZE_SCHEMA}.topdesk_incident_urgencies_raw",
    # },
    # {
    #     "name": "categories",
    #     "source": "topdesk",
    #     "url": f"{TOPDESK_BASE}/incidents/categories",
    #     "pagination": "SINGLE",
    #     "bronze_table": f"{CATALOG}.{BRONZE_SCHEMA}.topdesk_incident_categories_raw",
    # },
    # {
    #     "name": "subcategories",
    #     "source": "topdesk",
    #     "url": f"{TOPDESK_BASE}/incidents/subcategories",
    #     "pagination": "SINGLE",
    #     "bronze_table": f"{CATALOG}.{BRONZE_SCHEMA}.topdesk_incident_subcategories_raw",
    # },
    # {
    #     "name": "closure_codes",
    #     "source": "topdesk",
    #     "url": f"{TOPDESK_BASE}/incidents/closure_codes",
    #     "pagination": "SINGLE",
    #     "bronze_table": f"{CATALOG}.{BRONZE_SCHEMA}.topdesk_incident_closurecodes_raw",
    # },
    # {
    #     "name": "branches",
    #     "source": "topdesk",
    #     "url": f"{TOPDESK_BASE}/branches",
    #     "pagination": "SINGLE",
    #     "bronze_table": f"{CATALOG}.{BRONZE_SCHEMA}.topdesk_branches_raw",
    # },
    # {
    #     "name": "budgetholders",
    #     "source": "topdesk",
    #     "url": f"{TOPDESK_BASE}/budgetholders",
    #     "pagination": "SINGLE",
    #     "bronze_table": f"{CATALOG}.{BRONZE_SCHEMA}.topdesk_budgetholders_raw",
    # },
    # {
    #     "name": "operators",
    #     "source": "topdesk",
    #     "url": f"{TOPDESK_BASE}/operators",
    #     "pagination": "OFFSET",
    #     "bronze_table": f"{CATALOG}.{BRONZE_SCHEMA}.topdesk_operators_raw",
    # },
    # {
    #     "name": "persons",
    #     "source": "topdesk",
    #     "url": f"{TOPDESK_BASE}/persons",
    #     "pagination": "OFFSET",
    #     "incremental_field": "modificationDate",
    #     "bronze_table": f"{CATALOG}.{BRONZE_SCHEMA}.topdesk_persons_raw",
    # },
    # {
    #     "name": "operatorgroup",
    #     "source": "topdesk",
    #     "url": f"{TOPDESK_BASE}/operatorgroups",
    #     "pagination": "OFFSET",
    #     "bronze_table": f"{CATALOG}.{BRONZE_SCHEMA}.topdesk_operatorgroup_raw",
    # },
    # {
    #     "name": "impacts",
    #     "source": "topdesk",
    #     "url": f"{TOPDESK_BASE}/changes/impacts",
    #     "pagination": "SINGLE",
    #     "bronze_table": f"{CATALOG}.{BRONZE_SCHEMA}.topdesk_changes_impacts_raw",
    # },
    # {
    #     "name": "operatorchanges",
    #     "source": "topdesk",
    #     "url": f"{TOPDESK_BASE}/operatorChanges",
    #     "pagination": "NEXT",
    #     "incremental_field": "modificationDate",
    #     "bronze_table": f"{CATALOG}.{BRONZE_SCHEMA}.topdesk_operator_changes_raw",
    #     "params": {"fields": "id,incident.id,changeType,status,category.id,category.name,subcategory.id,processingStatus,modificationDate"},
    # },
    # {
    #     "name": "operatorchangeactivities",
    #     "source": "topdesk",
    #     "url": f"{TOPDESK_BASE}/operatorChangeActivities",
    #     "pagination": "NEXT",
    #     "incremental_field": "lastModificationDate",
    #     "bronze_table": f"{CATALOG}.{BRONZE_SCHEMA}.topdesk_operator_changeActivities_raw",
    #     "params": {"fields": "timeSpent,id,category.id,lastModificationDate"},
    # },
    # {
    #     "name": "changestatuses",
    #     "source": "topdesk",
    #     "url": f"{TOPDESK_BASE}/changes/changeStatuses",
    #     "pagination": "NEXT",
    #     "bronze_table": f"{CATALOG}.{BRONZE_SCHEMA}.topdesk_changestatuses_raw",
    # },
    # {
    #     "name": "changeactivitystatuses",
    #     "source": "topdesk",
    #     "url": f"{TOPDESK_BASE}/changes/activityStatuses",
    #     "pagination": "NEXT",
    #     "bronze_table": f"{CATALOG}.{BRONZE_SCHEMA}.topdesk_changeActivitiesStatuses_raw",
    # },
    # {
    #     "name": "services",
    #     "source": "topdesk",
    #     "url": f"{TOPDESK_BASE}/services",
    #     "pagination": "SINGLE",
    #     "bronze_table": f"{CATALOG}.{BRONZE_SCHEMA}.topdesk_services_raw",
    # },

    # ── TopDesk OData Reporting API ─────────────────────────
    # Note: TopDesk OData does NOT support $filter on modificationDate
    #       These endpoints always do a full load.
    # {
    #     "name": "change_time_registrations",
    #     "source": "topdesk",
    #     "url": f"{TOPDESK_ODATA_BASE}/ChangeTimeRegistrations",
    #     "pagination": "ODATA_NEXT",
    #     "bronze_table": f"{CATALOG}.{BRONZE_SCHEMA}.topdesk_change_time_registrations_raw",
    # },
    # {
    #     "name": "change_activity_time_registrations",
    #     "source": "topdesk",
    #     "url": f"{TOPDESK_ODATA_BASE}/ChangeActivityTimeRegistrations",
    #     "pagination": "ODATA_NEXT",
    #     "bronze_table": f"{CATALOG}.{BRONZE_SCHEMA}.topdesk_change_activity_time_registrations_raw",
    # },

    # ── Jira ────────────────────────────────────────────────
    # {
    #     "name": "jira_issues",
    #     "source": "jira",
    #     "url": "https://msfsits-jira.atlassian.net/rest/api/3/search/jql",
    #     "pagination": "JIRA_NEXT",
    #     "bronze_table": f"{CATALOG}.{BRONZE_SCHEMA}.jira_issues_raw",
    #     "params": {"jql": "project NOT IN (MSFTEMPL)", "fields": "*all", "maxResults": PAGE_SIZE},
    # },

    # ── Microsoft Entra ────────────────────────────────────
    # {
    #     "name": "entra_users",
    #     "source": "entra",
    #     "url": "https://graph.microsoft.com/v1.0/users",
    #     "pagination": "GRAPH_NEXT",
    #     "bronze_table": f"{CATALOG}.{BRONZE_SCHEMA}.entra_users_raw",
    #     "params": {
    #         "$filter": "endswith(userPrincipalName,'@sits.msf.org')",
    #         "$select": "displayName,mail,department",
    #         "$count": "true",
    #     },
    # },

    # ── Tempo ──────────────────────────────────────────────
    # {
    #     "name": "tempo_worklogs",
    #     "source": "tempo",
    #     "url": "https://api.eu.tempo.io/4/worklogs",
    #     "pagination": "TEMPO_NEXT",
    #     "bronze_table": f"{CATALOG}.{BRONZE_SCHEMA}.tempo_worklogs_raw",
    #     "params": {"limit": 1000},
    # },

    # ── SharePoint Files (via Graph API) ────────────────────
    # Requires Sites.Selected app permission + per-site grant.
    # All files from: sites/GRP-SITS-COSTRECOVERY / General/PowerBI Master Files/
    # {
    #     "name": "operator_cost",
    #     "source": "sharepoint",
    #     "site_path": _SP_SITE,
    #     "file_path": f"{_SP_DIR}/Operator_Cost.xlsx",
    #     "pagination": "SHAREPOINT_FILE",
    #     "bronze_table": f"{CATALOG}.{BRONZE_SCHEMA}.sharepoint_operator_cost_raw",
    # },
    # {
    #     "name": "exchange_rates",
    #     "source": "sharepoint",
    #     "site_path": _SP_SITE,
    #     "file_path": f"{_SP_DIR}/Exchange rates.xlsx",
    #     "pagination": "SHAREPOINT_FILE",
    #     "bronze_table": f"{CATALOG}.{BRONZE_SCHEMA}.sharepoint_exchange_rates_raw",
    # },
    # {
    #     "name": "cofund_project",
    #     "source": "sharepoint",
    #     "site_path": _SP_SITE,
    #     "file_path": f"{_SP_DIR}/CoFoundProjectTransformed.xlsx",
    #     "pagination": "SHAREPOINT_FILE",
    #     "bronze_table": f"{CATALOG}.{BRONZE_SCHEMA}.sharepoint_cofund_project_raw",
    # },
    # {
    #     "name": "capacity_billable",
    #     "source": "sharepoint",
    #     "site_path": _SP_SITE,
    #     "file_path": f"{_SP_DIR}/Capacity of Billable Resources.xlsx",
    #     "pagination": "SHAREPOINT_FILE",
    #     "bronze_table": f"{CATALOG}.{BRONZE_SCHEMA}.sharepoint_capacity_billable_raw",
    # },
    # {
    #     "name": "mainstream",
    #     "source": "sharepoint",
    #     "site_path": _SP_SITE,
    #     "file_path": f"{_SP_DIR}/2021 Q4 Mainstream.xlsx",
    #     "pagination": "SHAREPOINT_FILE",
    #     "bronze_table": f"{CATALOG}.{BRONZE_SCHEMA}.sharepoint_mainstream_raw",
    # },
    # {
    #     "name": "aliases",
    #     "source": "sharepoint",
    #     "site_path": _SP_SITE,
    #     "file_path": f"{_SP_DIR}/Aliases.xlsx",
    #     "pagination": "SHAREPOINT_FILE",
    #     "bronze_table": f"{CATALOG}.{BRONZE_SCHEMA}.sharepoint_aliases_raw",
    # },
]

In [0]:
# ============================================================
# AUTHENTICATION
# ============================================================

def get_graph_token() -> str:
    """Acquire an OAuth2 client-credentials token for Microsoft Graph."""
    resp = requests.post(
        f"https://login.microsoftonline.com/{tenant_id}/oauth2/v2.0/token",
        data={
            "client_id":     client_id,
            "client_secret": client_secret,
            "scope":         "https://graph.microsoft.com/.default",
            "grant_type":    "client_credentials",
        },
        timeout=REQUEST_TIMEOUT,
    )
    resp.raise_for_status()
    return resp.json()["access_token"]


def create_session(source: str) -> requests.Session:
    """Return a requests.Session with retry logic and source-appropriate auth."""
    session = requests.Session()
    retry = Retry(
        total=MAX_RETRIES,
        backoff_factor=2,
        status_forcelist=[429, 500, 502, 503, 504],
    )
    session.mount("https://", HTTPAdapter(max_retries=retry))

    if source == "topdesk":
        session.auth = (topdesk_username, topdesk_password)
    elif source == "jira":
        session.auth = (jira_username, jira_token)
    elif source in ("entra", "sharepoint"):
        session.headers.update({
            "Authorization":    f"Bearer {get_graph_token()}",
            "ConsistencyLevel": "eventual",
        })
    elif source == "tempo":
        session.headers["Authorization"] = f"Bearer {tempo_token}"
    else:
        raise ValueError(f"Unknown source: {source}")

    return session


# ============================================================
# TIMESTAMP UTILITIES
# ============================================================

def format_ts(dt: datetime) -> str:
    """Format a datetime as ISO 8601 with 'Z' suffix."""
    return dt.strftime("%Y-%m-%dT%H:%M:%S.%f") + "Z"


def parse_ts(value: Optional[str]) -> Optional[datetime]:
    """Parse an ISO 8601 timestamp string into a datetime object."""
    if not value:
        return None
    if value.endswith("Z"):
        value = value.replace("Z", "+00:00")
    return datetime.fromisoformat(value)

In [0]:
# ============================================================
# STATE MANAGEMENT & QUERY HELPERS
# ============================================================

def _pipeline_name(endpoint: dict) -> str:
    """Derive a pipeline name from the endpoint source."""
    return f"{endpoint['source']}_ingestion"


def ensure_state_table_exists():
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {STATE_TABLE} (
            pipeline STRING,
            endpoint STRING,
            last_run STRING
        ) USING DELTA
    """)


def get_last_run(endpoint: dict) -> Optional[str]:
    """Return the last successful run timestamp, or None on FULL mode / first run."""
    if LOAD_TYPE == "FULL":
        return None
    pname = _pipeline_name(endpoint)
    rows = spark.sql(f"""
        SELECT last_run FROM {STATE_TABLE}
        WHERE pipeline = '{pname}' AND endpoint = '{endpoint["name"]}'
        LIMIT 1
    """).collect()
    return rows[0]["last_run"] if rows else None


def update_last_run(endpoint: dict, ts: str):
    """Upsert the last_run timestamp in the state table."""
    pname = _pipeline_name(endpoint)
    spark.sql(f"""
        MERGE INTO {STATE_TABLE} t
        USING (SELECT '{pname}' AS pipeline, '{endpoint["name"]}' AS endpoint, '{ts}' AS last_run) s
        ON t.pipeline = s.pipeline AND t.endpoint = s.endpoint
        WHEN MATCHED THEN UPDATE SET t.last_run = s.last_run
        WHEN NOT MATCHED THEN INSERT *
    """)


def _build_incremental_filter(endpoint: dict, last_run: Optional[str]) -> Optional[str]:
    """Build an incremental filter string appropriate for the endpoint's protocol.

    TopDesk REST  \u2192 modificationDate>'2024-01-01T00:00:00.000000Z'
    OData         \u2192 modificationDate gt 2024-01-01T00:00:00Z  (seconds only)
    """
    inc_field = endpoint.get("incremental_field")
    if LOAD_TYPE == "FULL" or not last_run or not inc_field:
        return None
    if endpoint["pagination"] == "ODATA_NEXT":
        # OData rejects microsecond precision; strip to seconds
        ts = last_run.split(".")[0] + "Z" if "." in last_run else last_run
        return f"{inc_field} gt {ts}"
    return f"{inc_field}>'{last_run}'"


def _base_params(endpoint: dict, inc_filter: Optional[str] = None) -> dict:
    """Merge static endpoint params with an optional incremental filter."""
    params = dict(endpoint.get("params", {}))
    if inc_filter:
        if endpoint["pagination"] == "ODATA_NEXT":
            params["$filter"] = inc_filter
        else:
            params["query"] = inc_filter
    return params

In [0]:
# ============================================================
# INGESTION ENGINE
# ============================================================

def write_to_bronze(records: List[Dict], endpoint: dict, mode: str = "append") -> None:
    """Write a batch of records to the bronze Delta table."""
    now = datetime.now(timezone.utc).isoformat()
    rows = [Row(raw_json=json.dumps(rec), ingestion_time=now) for rec in records]
    df = spark.createDataFrame(rows)
    df.write.format("delta").mode(mode).saveAsTable(endpoint["bronze_table"])
    print(f"  \u2713 Written {len(records):,} records \u2192 {endpoint['bronze_table']}")


def ingest(endpoint: dict) -> tuple:
    """
    Fetch all records from an endpoint and write them to its bronze table.
    Returns (total_records, max_timestamp_str | None).
    """
    name       = endpoint["name"]
    source     = endpoint["source"]
    inc_field  = endpoint.get("incremental_field")
    pag        = endpoint["pagination"]

    last_run   = get_last_run(endpoint)
    inc_filter = _build_incremental_filter(endpoint, last_run)

    if inc_filter:
        print(f"  Filter: {inc_filter}")
    else:
        print(f"  Mode: FULL (no filter)")

    total       = 0
    batch       = []
    max_ts      = None
    first_write = True          # first batch of a FULL run uses overwrite

    # -- helpers local to this run --------------------------------
    def _track(rec):
        nonlocal max_ts
        if inc_field:
            val = rec.get(inc_field)
            if val:
                parsed = parse_ts(val)
                if parsed and (max_ts is None or parsed > max_ts):
                    max_ts = parsed

    def _flush():
        nonlocal first_write
        if batch:
            mode = "overwrite" if (LOAD_TYPE == "FULL" and first_write) else "append"
            write_to_bronze(batch, endpoint, mode)
            batch.clear()
            first_write = False
    # -------------------------------------------------------------

    with create_session(source) as session:

        # == SINGLE (no pagination) ==============================
        if pag == "SINGLE":
            params  = _base_params(endpoint, inc_filter)
            resp    = session.get(endpoint["url"], params=params, timeout=REQUEST_TIMEOUT)
            resp.raise_for_status()
            payload = resp.json()
            records = payload.get("results", payload) if isinstance(payload, dict) else payload
            for rec in records:
                _track(rec)
                batch.append(rec)
                total += 1

        # == OFFSET (start + page_size) ==========================
        elif pag == "OFFSET":
            offset = 0
            while True:
                params = _base_params(endpoint, inc_filter)
                params.update({"start": offset, "page_size": PAGE_SIZE})
                resp = session.get(endpoint["url"], params=params, timeout=REQUEST_TIMEOUT)
                resp.raise_for_status()
                if not resp.text.strip():
                    break
                data = resp.json()
                if not data:
                    break
                for rec in data:
                    _track(rec)
                    batch.append(rec)
                    total += 1
                if len(batch) >= BATCH_SIZE:
                    _flush()
                if len(data) < PAGE_SIZE:
                    break
                offset += PAGE_SIZE
                print(f"  Fetched {total:,}")

        # == NEXT (TopDesk next-link) ============================
        elif pag == "NEXT":
            url    = endpoint["url"]
            params = _base_params(endpoint, inc_filter)
            while url:
                resp = session.get(url, params=params, timeout=REQUEST_TIMEOUT)
                resp.raise_for_status()
                payload = resp.json()
                for rec in payload.get("results", []):
                    _track(rec)
                    batch.append(rec)
                    total += 1
                if len(batch) >= BATCH_SIZE:
                    _flush()
                print(f"  Fetched {total:,}")
                url    = payload.get("next")
                params = {}                         # next URL carries its own params

        # == JIRA_NEXT (Jira cursor pagination) ==================
        elif pag == "JIRA_NEXT":
            url    = endpoint["url"]
            params = _base_params(endpoint, inc_filter)
            token  = None
            while True:
                if token:
                    params["nextPageToken"] = token
                resp = session.get(url, params=params, timeout=REQUEST_TIMEOUT)
                if resp.status_code != 200:
                    print(resp.text)
                resp.raise_for_status()
                payload = resp.json()
                for issue in payload.get("issues", []):
                    _track(issue)
                    batch.append(issue)
                    total += 1
                if len(batch) >= BATCH_SIZE:
                    _flush()
                print(f"  Fetched {total:,}")
                token = payload.get("nextPageToken")
                if not token:
                    break

        # == GRAPH_NEXT / ODATA_NEXT (@odata.nextLink) ===========
        elif pag in ("GRAPH_NEXT", "ODATA_NEXT"):
            url    = endpoint["url"]
            params = _base_params(endpoint, inc_filter)
            while url:
                resp = session.get(url, params=params, timeout=REQUEST_TIMEOUT)
                resp.raise_for_status()
                payload = resp.json()
                for rec in payload.get("value", []):
                    _track(rec)
                    batch.append(rec)
                    total += 1
                if len(batch) >= BATCH_SIZE:
                    _flush()
                print(f"  Fetched {total:,}")
                url    = payload.get("@odata.nextLink")
                params = {}                         # nextLink carries its own query string

        # == TEMPO_NEXT (Tempo metadata.next) ====================
        elif pag == "TEMPO_NEXT":
            url    = endpoint["url"]
            params = _base_params(endpoint, inc_filter)
            while url:
                resp = session.get(url, params=params, timeout=REQUEST_TIMEOUT)
                resp.raise_for_status()
                payload = resp.json()
                for rec in payload.get("results", []):
                    _track(rec)
                    batch.append(rec)
                    total += 1
                if len(batch) >= BATCH_SIZE:
                    _flush()
                print(f"  Fetched {total:,}")
                url    = payload.get("metadata", {}).get("next")
                params = {}                         # next URL carries its own query string

        # == SHAREPOINT_FILE (Excel/CSV via Graph API) ===========
        elif pag == "SHAREPOINT_FILE":
            # 1. Resolve site ID from site_path
            site_url = f"https://graph.microsoft.com/v1.0/sites/{SHAREPOINT_HOST}:/{endpoint['site_path']}"
            site_resp = session.get(site_url, timeout=REQUEST_TIMEOUT)
            site_resp.raise_for_status()
            site_id = site_resp.json()["id"]
            print(f"  Site resolved: {site_id}")

            # 2. Download file content
            file_path = endpoint["file_path"]
            file_url  = f"https://graph.microsoft.com/v1.0/sites/{site_id}/drive/root:/{file_path}:/content"
            file_resp = session.get(file_url, timeout=60, allow_redirects=True)
            file_resp.raise_for_status()
            print(f"  Downloaded {len(file_resp.content):,} bytes")

            # 3. Parse file into a pandas DataFrame
            if file_path.endswith(".xlsx"):
                pdf = pd.read_excel(
                    io.BytesIO(file_resp.content),
                    engine="openpyxl",
                    sheet_name=endpoint.get("sheet_name", 0),
                )
            elif file_path.endswith(".csv"):
                pdf = pd.read_csv(io.BytesIO(file_resp.content))
            else:
                raise ValueError(f"Unsupported file format: {file_path}")

            # 4. Convert rows to JSON-safe dicts (handles NaN, dates, etc.)
            records_list = json.loads(pdf.to_json(orient="records", date_format="iso"))
            for rec in records_list:
                _track(rec)
                batch.append(rec)
                total += 1
                if len(batch) >= BATCH_SIZE:
                    _flush()
            print(f"  Parsed {total:,} rows from {file_path}")

        else:
            raise ValueError(f"Unknown pagination type: {pag}")

    _flush()
    return total, format_ts(max_ts) if max_ts else None

In [0]:
# ============================================================
# MAIN
# ============================================================

def main():
    print(f"{'=' * 60}")
    print(f"  BRONZE INGESTION \u2014 {LOAD_TYPE} mode")
    print(f"{'=' * 60}\n")

    ensure_state_table_exists()

    for ep in ENDPOINTS:
        print(f"\u25b8 {ep['name']}  ({ep['source']})")

        total, max_ts = ingest(ep)
        print(f"  Done: {total:,} records")

        if ep.get("incremental_field") and max_ts:
            update_last_run(ep, max_ts)
            print(f"  State updated \u2192 {max_ts}")

        print()

    print(f"{'=' * 60}")
    print("  INGESTION COMPLETE")
    print(f"{'=' * 60}")

In [0]:
if __name__ == "__main__":
    main()